In [ ]:
import copy
from IPython.display import Markdown, display, Video
import os

import numpy as np

import sys
sys.path.extend(["../../"])

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MultipleLocator, FormatStrFormatter
import imageio.v2 as imageio

from kmeans import Kmeans, assign_data_to_centroids, kmeans_loss_function
from kmeans_plotting import Plot1DKmeans, Plot2DKmeans
from sklearn.cluster import KMeans as KMeans_sklearn


In [ ]:
display(Markdown(open("../../_macros.md").read()))

# Clustering

Clustering refers to the set of techniques which allows to automatically group data into what are called clusters. There are many ways of implementing this process. From techniques that fix a maximum number of clusters, others that learn hierarchical clusters, and others that can learn the number of clusters automatically. 

Clustering is also grouped under a set of algorithms known as unsupervised learning, although I do not like this taxonomy. Also, many algorithms are not well known for being used as clustering algorithms (such as mixture models), but we can obtain clusters from their construction. 

Examples of algorithms that perform clustering include K-means, HDBSCAN, and Linear and non-linear mixture models, and their infinite generalization through Dirichlet processes. There might be plenty more of them.

This document describes K-means.

## Kmeans

K-means is an algorithm that assigns data into clusters based on computing a distance, with a predefined number of clusters, which are then learnt automatically.

While it has a connection with Probabilistic Machine Learning (more precisely, it is a degenerate version of a Gaussian Mixture Model fitted through Maximum Log-Likelihood), the usual more intuitive design and description of the algorithm does not have a direct and clear connection with any of the usual mathematical probabilistic tools, in contrast with linear or logistic regression.

However, one can arrive at the algorithm and show that it is just the Expectation Maximization algorithm assuming a Gaussian mixture model where the variances are not learnt and set to equal values, but it is not so direct. Another way of arriving at the algorithm is by the optimization of a loss function, which, due to the optimization variables being involved (some of them being discrete), requires a coordinate descent optimization procedure. To my knowledge, this loss function does not have a probabilistic interpretation, in contrast to the approach that starts from a Gaussian Mixture Model. I would also need to check whether both approaches lead to the same optimization function, which I think it doesnt.

Since the loss function of the algorithm is non-convex, there are many local minima at which we can arrive. We will see that, in fact, initialization is a very important step in this algorithm.

The basic idea behind this clustering algorithm is shown in the following video. Assume we have the following data, and we wish to find groups within the data automatically.

In [ ]:
np.random.seed(1)

N_points = 400

## probability p(c)
p = [0.33,0.33,0.34]
c = [0,1,2]

## probability p(x|c)
var1 = 0.02
var2 = 0.04
var3 = 0.01
p_xc = {
    'c_0' : {
        'mu' : np.array([0,0]),
        'cov' : var1*np.eye(2),
        },
    'c_1' : {
        'mu' : np.array([0.5,0.5]),
        'cov' : var2*np.eye(2),
        },
    'c_2' : {
        'mu' : np.array([0,0.5]),
        'cov' : var3*np.eye(2),
        }   
}

## sample cluster assignments
cluster = np.random.choice(c, size=N_points, p=p)

## 
X = np.zeros((N_points,2), dtype = np.float32)
counter = 0
X_clust = {}
for _c in c:
    num_c = np.sum(cluster==_c)
    
    mu  = p_xc[f'c_{_c}']['mu']
    cov = p_xc[f'c_{_c}']['cov']
    
    _x = np.random.multivariate_normal(mean=mu, cov=cov, size = num_c)
    
    X[counter:counter + num_c] = _x
    X_clust[_c] = _x
    
    counter += num_c

fig, (ax1,ax2) = plt.subplots(1,2, figsize = (10,5))
ax1.plot(X[:,0],X[:,1], 'x', color = 'k')
ax1.set_xlabel(r"$x_1$")
ax1.set_ylabel(r"$x_2$")
ax1.set_title("Unlabelled Data")

for _c in c:
    _x = X_clust[_c]
    ax2.plot(_x[:,0],_x[:,1], 'x', color = f"C{_c}")
    ax2.set_xlabel(r"$x_1$")
    ax2.set_ylabel(r"$x_2$")
    ax2.set_title("True Cluster Assignment")


## shuffle data
for i in range(10):
    np.random.shuffle(X)

Each cluster is represented by a point called a centroid, and data is iteratively assigned to clusters, with new centroid recomputation after data assignment until convergence. Depending on the number of centroids, we arrive at different solutions. Different initializations also lead to different solutions. Most of the time, we do not know how many clusters we need to use, unless we can visualize the data or we have some domain knowledge.

### Examples

#### 4 clusters

In [ ]:
interactive_plot = False

if interactive_plot:
    %matplotlib tk
else:
    %matplotlib inline
    
plotter = Plot2DKmeans(
                        video = True, 
                        interactive_plot = interactive_plot, 
                        sleep_time = 0.1, 
                        draw_voronoi = True,
                      )


kmeans = Kmeans(num_centroids = 4, plotter = plotter)
kmeans.run(X, num_iters = 20, seed = 1)
plotter.show_video()


In [ ]:
interactive_plot = False

if interactive_plot:
    %matplotlib tk
else:
    %matplotlib inline
    
plotter = Plot2DKmeans(
                        video = True, 
                        interactive_plot = interactive_plot, 
                        sleep_time = 0.1, 
                        draw_voronoi = True,
                      )

kmeans = Kmeans(num_centroids = 4, plotter = plotter)
kmeans.run(X, num_iters = 20, seed = 10)

plotter.show_video()

#### 3 clusters

In [ ]:
interactive_plot = False

if interactive_plot:
    %matplotlib tk
else:
    %matplotlib inline
    
plotter = Plot2DKmeans(
                        video = True, 
                        interactive_plot = interactive_plot, 
                        sleep_time = 0.1, 
                        draw_voronoi = True,
                      )

kmeans = Kmeans(num_centroids = 3, plotter = plotter)
kmeans.run(X, num_iters = 20, seed = 1)

plotter.show_video()

In [ ]:
interactive_plot = False

if interactive_plot:
    %matplotlib tk
else:
    %matplotlib inline
    
plotter = Plot2DKmeans(
                        video = True, 
                        interactive_plot = interactive_plot, 
                        sleep_time = 0.1, 
                        draw_voronoi = True,
                      )

kmeans = Kmeans(num_centroids = 3, plotter = plotter)
kmeans.run(X, num_iters = 20, seed = 10)

plotter.show_video()

#### 10 clusters

In [ ]:
interactive_plot = False

if interactive_plot:
    %matplotlib tk
else:
    %matplotlib inline
    
plotter = Plot2DKmeans(
                        video = True, 
                        interactive_plot = interactive_plot, 
                        sleep_time = 0.1, 
                        draw_voronoi = True,
                      )

kmeans = Kmeans(num_centroids = 10, plotter = plotter)
kmeans.run(X, num_iters = 20, seed = 1)

plotter.show_video()

In [ ]:
interactive_plot = False

if interactive_plot:
    %matplotlib tk
else:
    %matplotlib inline
    
plotter = Plot2DKmeans(
                        video = True, 
                        interactive_plot = interactive_plot, 
                        sleep_time = 0.1, 
                        draw_voronoi = True,
                      )

kmeans = Kmeans(num_centroids = 10, plotter = plotter)
kmeans.run(X, num_iters = 40, seed = 10)

plotter.show_video()

The k-means algorithm is, obviously, independent of the data dimension. We can run it on 1-dimensional data and display clusters, but it is not as visual.

Consider the following data, again drawn using a mixture distribution

In [ ]:
np.random.seed(1)

N_points = 50

## probability p(c)
p = [0.33,0.33,0.34]
c = [0,1,2]

## probability p(x|c)
var1 = 0.1
var2 = 0.01
var3 = 0.01
p_xc = {
    'c_0' : {
        'mu' : np.array([0]),
        'cov' : var1*np.eye(1),
        },
    'c_1' : {
        'mu' : np.array([0.2]),
        'cov' : var2*np.eye(1),
        },
    'c_2' : {
        'mu' : np.array([0.35]),
        'cov' : var3*np.eye(1),
        }   
}

## sample cluster assignments
cluster = np.random.choice(c, size=N_points, p=p)

## 
X = np.zeros((N_points,1), dtype = np.float32)
counter = 0
X_clust = {}
for _c in c:
    num_c = np.sum(cluster==_c)
    
    mu  = p_xc[f'c_{_c}']['mu']
    cov = p_xc[f'c_{_c}']['cov']
    
    _x = np.random.multivariate_normal(mean=mu, cov=cov, size = num_c)
    
    X[counter:counter + num_c] = _x
    X_clust[_c] = _x
    
    counter += num_c

fig, (ax1,ax2) = plt.subplots(1,2, figsize = (10,5))
ax1.plot(X[:,0],np.zeros_like(X), 'x', color = 'k')
ax1.set_xlabel(r"$x_1$")
ax1.set_ylabel(r"$x_2$")
ax1.set_title("Unlabelled Data")

for _c in c:
    _x = X_clust[_c]
    ax2.plot(_x[:,0],np.zeros_like(_x), 'x', color = f"C{_c}")
    ax2.set_xlabel(r"$x_1$")
    ax2.set_ylabel(r"$x_2$")
    ax2.set_title("True Cluster Assignment")


## shuffle data
for i in range(10):
    np.random.shuffle(X)

In [ ]:
interactive_plot = False

if interactive_plot:
    %matplotlib tk
else:
    %matplotlib inline
    
plotter = Plot1DKmeans(
                        video = True, 
                        interactive_plot = interactive_plot, 
                        sleep_time = 0.1, 
                        draw_voronoi = True,
                      )

kmeans = Kmeans(num_centroids = 3, plotter = plotter)
kmeans.run(X, num_iters = 20, seed = 10)

plotter.show_video()

### Algorithm

Usually, in machine learning, we start by defining the functional form of the model, the loss function, and the associated optimization procedure.

For example in linear regression, the model is the set of all possible hyperplanes parameterized by $\Wmat$, the associated loss function can be the squared loss, and the optimization proceedure could be exact method (known also as least squares due to its connection to how Linear Systems solutions can be obtained when exact solution cannot be computed), gradient descent, steepest gradient descent, or coordinate descent among others.

K-means is rather more intuitive to start with the algorithm, since arriving at this algorithm from the model form and loss function does not provide an intuitive understanding of the algorithm.

The K-means algorithm has different steps.

* 1.  Initialization. Select the number of centroids to obtain and initialize them to any value. This could be done by selecting random points from the dataset, by selecting random points from the space of points, or by using advanced techniques such as the k-means++ algorithm.

After initialization, iterate the following steps:

* 2. Assign each data point to its closest centroid wrt some norm. The $L_2$ norm is usually used and is the one that connects with the Gaussian Mixture Model.
* 3. Recompute centroids by the sample mean of the data points assigned to each of the clusters.

Assume we have $N$ datapoints, with $K$ clusters $S=\{S_1,\dots,S_K\}$. Each point is denoted by $\xvec_n \in \mathbb{R}^d$, each cluster centroid by $\muvec_k$, and $c_n$ denotes the cluster being assigned to each $\xvec_n$. We denote the whole set of centroids by $\{\muvec_k\}_{k=1}^{K}$ and the set of cluster assignments by $c = \{c_1,c_2,\dots,c_n\}$.


Mathematically, we have:

* Step 2:

* 
$$
\begin{split}
c_n = \underset{k}{\text{argmin}}\mid\mid \xvec_n - \muvec_k \mid\mid^2
\end{split}
$$

* Step 3: For each cluster $c$:

$$
\begin{split}
\muvec_k = \frac{1}{\mid S_k \mid}\sum_{\xvec_i \in S_k} \xvec_i
\end{split}
$$

**Algorithm derivation**: We can arrive at this algorithm via two different paths (to my knowledge).
* 1. Through maximum likelihood of a Gaussian Mixture model with non-learnable variances, optimized with the EM algorithm, which is a coordinate descent algorithm over parameter values (step 3) and probability distributions (step 2)
* 2. By defining a loss function which is optimized through coordinate descent over cluster assignments (step 2= and centroid computations (step 3). We will refer to this approach as the loss function approach, and it is the one we study now.

As we see above, step 2 is clearly a minimization problem. Step 3 is the solution to the other minimization problem that arises when optimizing over the other coordinates, the centroid values. Due to the form of the solution (which is unique), we can easily observe that step 3's minimization function is convex. In other words, given fixed cluster assignments $c_n$, optimizing over $\{\muvec_k\}_{k=1}^{K}$ is a convex minimization problem whose minimum is the sample mean of points assigned to the cluster.

### Loss function approach

The non-probabilistic approach to k-means starts by defining the following loss function to be optimized. This loss function is also known as the within-cluster sum of squares, which is basically the variance. In other words, we wish to minimize the variance from the clusters generated, which makes sense because the smaller the variance, the more concentrated the points are in each cluster, which is equivalent to maximizing the sum of squared deviations between points in different clusters.

$$
\begin{split}
L(c,\{\muvec_k\}_{k=1}^{K},\Xmat) = \sum^K_{k=1}\sum_{\xvec_i \in S_k}\mid\mid \xvec_i - \muvec_k \mid\mid^2
\end{split}
$$

The optimization procedure seeks to find both cluster centroids $\muvec_c$ and optimal cluster assignments $c_n$. This means the optimization is defined by:

$$
\begin{split}
\underset{c,\{\muvec_k\}_{k=1}^{K}}{\text{argmin}}\,\,L(c,\{\muvec_k\}_{k=1}^{K},\Xmat)
\end{split}
$$

Minimizing this function cannot be done by exact methods nor by gradient descent, because we wish to obtain optimal cluster assignments $c_n$ which are discrete variables, and so the gradient is not defined. It turns out that a coordinate descent approach can be used to solve this problem and lead to the algorithm described above; see: https://wei2624.github.io/MachineLearning/usv_kmeans/


#### Functional form of the loss function.

The loss function from the loss function approach is non-convex and mixes both continuous variables, the centroids $\muvec_k$, and discrete variables, the cluster data point assignments $c_n$. To explicitly state this dependence, we can rewrite the loss function in the usual form:

$$
\begin{split}
L(c,\{\muvec_k\}_{k=1}^{K},\Xmat) = \sum^K_{k=1}\sum_{n=1}^N r_{nk}\mid\mid \xvec_n - \muvec_k \mid\mid^2
\end{split}
$$

where $r_{nk}$ is an indicator variable such that:

$$
\begin{split}
r_{nk} = \begin{cases}
1, c_n = k\\
0, c_n \neq k
\end{cases}
\end{split}
$$

Note that here we have $K$ continous variables $\muvec_k \in \mathbb{R}^d$ and $N$ discrete variables $c_n \in \mathbb{N}$. Thus, the loss function can only be plotted when some variables are fixed and others vary, as we have done with linear regression. Since cluster assignments are discrete, the optimization function is weird to visualize, and so we will be showing the loss function varying centroid values. 

Note that, whenever a cluster value changes, it changes the cluster assignments, and thus we will see that the loss function has non-smooth changes, which correspond to when a cluster assignment has changed. However, within the same cluster assignment, we will see that the loss function is convex and there is a unique minimum of this loss within a cluster assignment.

##### Loss function for 1 dimensional data

**2 clusters N = 3**

In [ ]:
np.random.seed(1)

N_points = 3

## probability p(c)
p = [0.33,0.33,0.34]
c = [0,1,2]

## probability p(x|c)
var1 = 0.1
var2 = 0.01
var3 = 0.01
p_xc = {
    'c_0' : {
        'mu' : np.array([0]),
        'cov' : var1*np.eye(1),
        },
    'c_1' : {
        'mu' : np.array([3]),
        'cov' : var2*np.eye(1),
        },
    'c_2' : {
        'mu' : np.array([5]),
        'cov' : var3*np.eye(1),
        }   
}

## sample cluster assignments
cluster = np.random.choice(c, size=N_points, p=p)

## 
X = np.zeros((N_points,1), dtype = np.float32)
counter = 0
X_clust = {}
for _c in c:
    num_c = np.sum(cluster==_c)
    
    mu  = p_xc[f'c_{_c}']['mu']
    cov = p_xc[f'c_{_c}']['cov']
    
    _x = np.random.multivariate_normal(mean=mu, cov=cov, size = num_c)
    
    X[counter:counter + num_c] = _x
    X_clust[_c] = _x
    
    counter += num_c

fig, (ax1,ax2) = plt.subplots(1,2, figsize = (10,5))
ax1.plot(X[:,0],np.zeros_like(X), 'x', color = 'k')
ax1.set_xlabel(r"$x_1$")
ax1.set_title("Unlabelled Data")

for _c in c:
    _x = X_clust[_c]
    ax2.plot(_x[:,0],np.zeros_like(_x), 'x', color = f"C{_c}")
    ax2.set_xlabel(r"$x_1$")
    ax2.set_title("True Cluster Assignment")


## shuffle data
for i in range(10):
    np.random.shuffle(X)

In [ ]:
interactive_plot = False

if interactive_plot:
    %matplotlib tk
else:
    %matplotlib inline
    
plotter = Plot1DKmeans(
                        video = True, 
                        interactive_plot = interactive_plot, 
                        sleep_time = 0.1, 
                        draw_voronoi = True,
                      )

kmeans = Kmeans(num_centroids = 2, plotter = plotter)
kmeans.initialize_centroids(X, seed = 1) 


In [ ]:
# video creation
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")  

# Plotting specifications
N_grid = 40
grid_lim_l = -10
grid_lim_u = 10
c_coord_range = np.reshape(np.linspace(grid_lim_l,grid_lim_u,N_grid),(N_grid,1))
vor_grid_xx = np.reshape( np.linspace(grid_lim_l, grid_lim_u, 500), (500,1))

fig_1, ax_list_1 = plt.subplots(kmeans.num_centroids,2, figsize = (10,5))

## Locally convex vs wthing assginment change
locally_convex = False
true_centroids_assigned, true_X_assigned = kmeans._assign_data_to_centroids(X)

## ====================
## Parameter to vary ##
## Centroid and coordinate

## to separate the cost function within the different assignments
assigned_regions = {}

# to keep loss
loss_acc = {}

# to keep region colors
region_colors = {}

for cent2change in range(kmeans.num_centroids):
    
    ## to separate the cost function within the different assignments
    assigned_regions[cent2change] = []

    ## Get centroids from the algorithm
    centroids = kmeans.centroids

    # to keep loss
    loss_acc[cent2change] = []
    
    for _c in c_coord_range:
        centroids[cent2change,:] = _c

        if locally_convex:
            centroids_assigned, X_assigned = true_centroids_assigned, true_X_assigned 
        else:
            centroids_assigned, X_assigned = kmeans._assign_data_to_centroids(X, centroids = centroids)

        assigned_regions[cent2change].append(tuple(centroids_assigned))

        loss = kmeans_loss_function(X, centroids = centroids, X_assigned = X_assigned)
        loss_acc[cent2change].append(loss)


    unique_assignments = list(set(assigned_regions[cent2change]))
    region_colors[cent2change] = [f"C{unique_assignments.index(a)}" for a in assigned_regions[cent2change]]
   
    ## ==================
    ## Draw loss function
    ax_list_1[cent2change][0].plot(c_coord_range, loss_acc[cent2change], c = 'C0')
    ax_list_1[cent2change][0].scatter(c_coord_range, loss_acc[cent2change], c = region_colors[cent2change])

    ax_list_1[cent2change][0].set_xlabel(f"Centroid {cent2change}")
    
       
## Draw how assignment changes and how it affecst loss function
for i in range(len(c_coord_range)):
    for cent2change in range(kmeans.num_centroids): 
        
        # get associated coordinates, loss and regions per centroid.
        _c,_loss,_cent_assigned = c_coord_range[i],loss_acc[cent2change][i],assigned_regions[cent2change][i]
        
        ax_list_1[cent2change][1].cla()

        # draw data assignment
        colors_data = [f"C{i}" for i in _cent_assigned]     
        ax_list_1[cent2change][1].scatter(X, np.zeros_like(X), marker = 'x',  c = colors_data)

        # highlight loss
        ax_list_1[cent2change][0].plot( _c, _loss, 'o' ,c = 'k')#, markersize =10)
        
        # remove previous highlighted
        if i != 0:
            ax_list_1[cent2change][0].plot(  c_coord_range[i-1], loss_acc[cent2change][i-1], 'o' , c = region_colors[cent2change][i-1])

        ## Get centroids from the algorithm
        centroids = kmeans.centroids

        # centroid
        centroids[cent2change,:] = _c

        ## draw centroid
        for idx,cet in enumerate(centroids):
            ## plot centroids
            ax_list_1[cent2change][1].plot(cet,0.0, 'o', color = f'C{idx}')

        ## draw voronoi regions
        voronoi_regions, _ = assign_data_to_centroids(vor_grid_xx , centroids)
        voronoi_regions_limits = np.where(np.diff(voronoi_regions) != 0)[0] + 1

        for c in voronoi_regions_limits:
            ax_list_1[cent2change][1].axvline(vor_grid_xx[c], color='k', linestyle="--")
            
        # fix xlim
        ax_list_1[cent2change][1].set_xlim([grid_lim_l, grid_lim_u])
        

    fig_1.canvas.draw()
    frame = np.asarray(fig_1.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)

writer.close()     

display(Video(data=video_filename, embed=True))
os.remove(video_filename)
plt.close(fig_1)

In [ ]:
# video creation
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")  

# Plotting specifications
N_grid = 40
grid_lim_l = -10
grid_lim_u = 10
c_coord_range = np.reshape(np.linspace(grid_lim_l,grid_lim_u,N_grid),(N_grid,1))
vor_grid_xx = np.reshape( np.linspace(grid_lim_l, grid_lim_u, 500), (500,1))

fig_1, ax_list_1 = plt.subplots(kmeans.num_centroids,2, figsize = (10,5))

## Locally convex vs wthing assginment change
locally_convex = True
true_centroids_assigned, true_X_assigned = kmeans._assign_data_to_centroids(X)

## ====================
## Parameter to vary ##
## Centroid and coordinate

## to separate the cost function within the different assignments
assigned_regions = {}

# to keep loss
loss_acc = {}

# to keep region colors
region_colors = {}

for cent2change in range(kmeans.num_centroids):
    
    ## to separate the cost function within the different assignments
    assigned_regions[cent2change] = []

    ## Get centroids from the algorithm
    centroids = kmeans.centroids

    # to keep loss
    loss_acc[cent2change] = []
    
    for _c in c_coord_range:
        centroids[cent2change,:] = _c

        if locally_convex:
            centroids_assigned, X_assigned = true_centroids_assigned, true_X_assigned 
        else:
            centroids_assigned, X_assigned = kmeans._assign_data_to_centroids(X, centroids = centroids)

        assigned_regions[cent2change].append(tuple(centroids_assigned))

        loss = kmeans_loss_function(X, centroids = centroids, X_assigned = X_assigned)
        loss_acc[cent2change].append(loss)


    unique_assignments = list(set(assigned_regions[cent2change]))
    region_colors[cent2change] = [f"C{unique_assignments.index(a)}" for a in assigned_regions[cent2change]]
   
    ## ==================
    ## Draw loss function
    ax_list_1[cent2change][0].plot(c_coord_range, loss_acc[cent2change], c = 'C0')
    ax_list_1[cent2change][0].scatter(c_coord_range, loss_acc[cent2change], c = region_colors[cent2change])

    ax_list_1[cent2change][0].set_xlabel(f"Centroid {cent2change}")
    
       
## Draw how assignment changes and how it affecst loss function
for i in range(len(c_coord_range)):
    for cent2change in range(kmeans.num_centroids): 
        
        # get associated coordinates, loss and regions per centroid.
        _c,_loss,_cent_assigned = c_coord_range[i],loss_acc[cent2change][i],assigned_regions[cent2change][i]
        
        ax_list_1[cent2change][1].cla()

        # draw data assignment
        colors_data = [f"C{i}" for i in _cent_assigned]     
        ax_list_1[cent2change][1].scatter(X, np.zeros_like(X), marker = 'x',  c = colors_data)

        # highlight loss
        ax_list_1[cent2change][0].plot( _c, _loss, 'o' ,c = 'k')#, markersize =10)
        
        # remove previous highlighted
        if i != 0:
            ax_list_1[cent2change][0].plot(  c_coord_range[i-1], loss_acc[cent2change][i-1], 'o' , c = region_colors[cent2change][i-1])

        ## Get centroids from the algorithm
        centroids = kmeans.centroids

        # centroid
        centroids[cent2change,:] = _c

        ## draw centroid
        for idx,cet in enumerate(centroids):
            ## plot centroids
            ax_list_1[cent2change][1].plot(cet,0.0, 'o', color = f'C{idx}')

        ## draw voronoi regions
        voronoi_regions, _ = assign_data_to_centroids(vor_grid_xx , centroids)
        voronoi_regions_limits = np.where(np.diff(voronoi_regions) != 0)[0] + 1

        for c in voronoi_regions_limits:
            ax_list_1[cent2change][1].axvline(vor_grid_xx[c], color='k', linestyle="--")
            
        # fix xlim
        ax_list_1[cent2change][1].set_xlim([grid_lim_l, grid_lim_u])
        

    fig_1.canvas.draw()
    frame = np.asarray(fig_1.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)

writer.close()     

display(Video(data=video_filename, embed=True))
os.remove(video_filename)
plt.close(fig_1)

In [ ]:
N_grid = 100

grid_lim_l_c0 = -10
grid_lim_u_c0 = 10

grid_lim_l_c1 = 3.8
grid_lim_u_c1 = 4.9
cA_coord_mesh,cB_coord_mesh = np.meshgrid(np.linspace(grid_lim_l_c0,grid_lim_u_c0,N_grid),np.linspace(grid_lim_l_c1,grid_lim_u_c1,N_grid))
c_mesh = np.hstack((np.reshape(cA_coord_mesh, (N_grid**2,1)),np.reshape(cB_coord_mesh, (N_grid**2,1))))
c_mesh = c_mesh[:,:, np.newaxis]


fig = plt.figure(figsize=(20, 10))
ax1 = fig.add_subplot(1, 2, 1) 
ax2 = fig.add_subplot(1, 2, 2, projection='3d')  
ax2.view_init(elev=30, azim=-90)

ax_list = [ax1, ax2]

## Get parameters and/or variables which are fixed 
locally_convex = False
true_centroids_assigned, true_X_assigned  = kmeans._assign_data_to_centroids(X)


## ====================
## Parameter to vary ##
# both clusters at the same time
loss_acc = []
for centroids in c_mesh:
    
    # print(c_mesh)
    
    if locally_convex:
        centroids_assigned, X_assigned = true_centroids_assigned, true_X_assigned 
    else:
        centroids_assigned, X_assigned = kmeans._assign_data_to_centroids(X, centroids = centroids)

    loss = kmeans.loss_function(X, centroids = centroids, X_assigned = X_assigned)
    loss_acc.append(loss)

loss_acc_mesh = np.reshape(np.array(loss_acc), (N_grid,N_grid))

ax_list[0].contourf(cA_coord_mesh, cB_coord_mesh, loss_acc_mesh, cmap = plt.cm.get_cmap("Oranges_r"), levels = 30)
ax_list[1].plot_surface(cA_coord_mesh, cB_coord_mesh, loss_acc_mesh, cmap = plt.cm.get_cmap("Oranges_r"))

ax_list[0].set_xlabel("centroid 0")
ax_list[0].set_ylabel("centroid 1")

ax_list[1].set_xlabel("centroid 0")
ax_list[1].set_ylabel("centroid 1")


**2 clusters N = 8**

In [ ]:
np.random.seed(1)

N_points = 8

## probability p(c)
p = [0.33,0.33,0.34]
c = [0,1,2]

## probability p(x|c)
var1 = 0.1
var2 = 0.1
var3 = 0.1
p_xc = {
    'c_0' : {
        'mu' : np.array([0]),
        'cov' : var1*np.eye(1),
        },
    'c_1' : {
        'mu' : np.array([3]),
        'cov' : var2*np.eye(1),
        },
    'c_2' : {
        'mu' : np.array([5]),
        'cov' : var3*np.eye(1),
        }   
}

## sample cluster assignments
cluster = np.random.choice(c, size=N_points, p=p)

## 
X = np.zeros((N_points,1), dtype = np.float32)
counter = 0
X_clust = {}
for _c in c:
    num_c = np.sum(cluster==_c)
    
    mu  = p_xc[f'c_{_c}']['mu']
    cov = p_xc[f'c_{_c}']['cov']
    
    _x = np.random.multivariate_normal(mean=mu, cov=cov, size = num_c)
    
    X[counter:counter + num_c] = _x
    X_clust[_c] = _x
    
    counter += num_c

fig, (ax1,ax2) = plt.subplots(1,2, figsize = (10,5))
ax1.plot(X[:,0],np.zeros_like(X), 'x', color = 'k')
ax1.set_xlabel(r"$x_1$")
ax1.set_title("Unlabelled Data")

for _c in c:
    _x = X_clust[_c]
    ax2.plot(_x[:,0],np.zeros_like(_x), 'x', color = f"C{_c}")
    ax2.set_xlabel(r"$x_1$")
    ax2.set_title("True Cluster Assignment")


## shuffle data
for i in range(10):
    np.random.shuffle(X)

In [ ]:
interactive_plot = False

if interactive_plot:
    %matplotlib tk
else:
    %matplotlib inline
    
plotter = Plot1DKmeans(
                        video = True, 
                        interactive_plot = interactive_plot, 
                        sleep_time = 0.1, 
                        draw_voronoi = True,
                      )

kmeans = Kmeans(num_centroids = 2, plotter = plotter)
kmeans.initialize_centroids(X, seed = 1) 

In [ ]:
# video creation
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")  

# Plotting specifications
N_grid = 50
grid_lim_l = -10
grid_lim_u = 10
c_coord_range = np.reshape(np.linspace(grid_lim_l,grid_lim_u,N_grid),(N_grid,1))
vor_grid_xx = np.reshape( np.linspace(grid_lim_l, grid_lim_u, 500), (500,1))

fig_1, ax_list_1 = plt.subplots(kmeans.num_centroids,2, figsize = (10,5))

## Locally convex vs wthing assginment change
locally_convex = False
true_centroids_assigned, true_X_assigned = kmeans._assign_data_to_centroids(X)

## ====================
## Parameter to vary ##
## Centroid and coordinate

## to separate the cost function within the different assignments
assigned_regions = {}

# to keep loss
loss_acc = {}

# to keep region colors
region_colors = {}

for cent2change in range(kmeans.num_centroids):
    
    ## to separate the cost function within the different assignments
    assigned_regions[cent2change] = []

    ## Get centroids from the algorithm
    centroids = kmeans.centroids

    # to keep loss
    loss_acc[cent2change] = []
    
    for _c in c_coord_range:
        centroids[cent2change,:] = _c

        if locally_convex:
            centroids_assigned, X_assigned = true_centroids_assigned, true_X_assigned 
        else:
            centroids_assigned, X_assigned = kmeans._assign_data_to_centroids(X, centroids = centroids)

        assigned_regions[cent2change].append(tuple(centroids_assigned))

        loss = kmeans_loss_function(X, centroids = centroids, X_assigned = X_assigned)
        loss_acc[cent2change].append(loss)


    unique_assignments = list(set(assigned_regions[cent2change]))
    region_colors[cent2change] = [f"C{unique_assignments.index(a)}" for a in assigned_regions[cent2change]]
   
    ## ==================
    ## Draw loss function
    ax_list_1[cent2change][0].plot(c_coord_range, loss_acc[cent2change], c = 'C0')
    ax_list_1[cent2change][0].scatter(c_coord_range, loss_acc[cent2change], c = region_colors[cent2change])

    ax_list_1[cent2change][0].set_xlabel(f"Centroid {cent2change}")
    
       
## Draw how assignment changes and how it affecst loss function
for i in range(len(c_coord_range)):
    for cent2change in range(kmeans.num_centroids): 
        
        # get associated coordinates, loss and regions per centroid.
        _c,_loss,_cent_assigned = c_coord_range[i],loss_acc[cent2change][i],assigned_regions[cent2change][i]
        
        ax_list_1[cent2change][1].cla()

        # draw data assignment
        colors_data = [f"C{i}" for i in _cent_assigned]     
        ax_list_1[cent2change][1].scatter(X, np.zeros_like(X), marker = 'x',  c = colors_data)

        # highlight loss
        ax_list_1[cent2change][0].plot( _c, _loss, 'o' ,c = 'k')
        
        # remove previous highlighted
        if i != 0:
            ax_list_1[cent2change][0].plot(  c_coord_range[i-1], loss_acc[cent2change][i-1], 'o' , c = region_colors[cent2change][i-1])

        ## Get centroids from the algorithm
        centroids = kmeans.centroids

        # centroid
        centroids[cent2change,:] = _c

        ## draw centroid
        for idx,cet in enumerate(centroids):
            ## plot centroids
            ax_list_1[cent2change][1].plot(cet,0.0, 'o', color = f'C{idx}')

        ## draw voronoi regions
        voronoi_regions, _ = assign_data_to_centroids(vor_grid_xx , centroids)
        voronoi_regions_limits = np.where(np.diff(voronoi_regions) != 0)[0] + 1

        for c in voronoi_regions_limits:
            ax_list_1[cent2change][1].axvline(vor_grid_xx[c], color='k', linestyle="--")
            
        # fix xlim
        ax_list_1[cent2change][1].set_xlim([grid_lim_l, grid_lim_u])
        

    fig_1.canvas.draw()
    frame = np.asarray(fig_1.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)

writer.close()     

display(Video(data=video_filename, embed=True))
os.remove(video_filename)
plt.close(fig_1)

In [ ]:
# video creation
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")  

# Plotting specifications
N_grid = 50
grid_lim_l = -10
grid_lim_u = 10
c_coord_range = np.reshape(np.linspace(grid_lim_l,grid_lim_u,N_grid),(N_grid,1))
vor_grid_xx = np.reshape( np.linspace(grid_lim_l, grid_lim_u, 500), (500,1))

fig_1, ax_list_1 = plt.subplots(kmeans.num_centroids,2, figsize = (10,5))

## Locally convex vs wthing assginment change
locally_convex = True
true_centroids_assigned, true_X_assigned = kmeans._assign_data_to_centroids(X)

## ====================
## Parameter to vary ##
## Centroid and coordinate

## to separate the cost function within the different assignments
assigned_regions = {}

# to keep loss
loss_acc = {}

# to keep region colors
region_colors = {}

for cent2change in range(kmeans.num_centroids):
    
    ## to separate the cost function within the different assignments
    assigned_regions[cent2change] = []

    ## Get centroids from the algorithm
    centroids = kmeans.centroids

    # to keep loss
    loss_acc[cent2change] = []
    
    for _c in c_coord_range:
        centroids[cent2change,:] = _c

        if locally_convex:
            centroids_assigned, X_assigned = true_centroids_assigned, true_X_assigned 
        else:
            centroids_assigned, X_assigned = kmeans._assign_data_to_centroids(X, centroids = centroids)

        assigned_regions[cent2change].append(tuple(centroids_assigned))

        loss = kmeans_loss_function(X, centroids = centroids, X_assigned = X_assigned)
        loss_acc[cent2change].append(loss)


    unique_assignments = list(set(assigned_regions[cent2change]))
    region_colors[cent2change] = [f"C{unique_assignments.index(a)}" for a in assigned_regions[cent2change]]
   
    ## ==================
    ## Draw loss function
    ax_list_1[cent2change][0].plot(c_coord_range, loss_acc[cent2change], c = 'C0')
    ax_list_1[cent2change][0].scatter(c_coord_range, loss_acc[cent2change], c = region_colors[cent2change])

    ax_list_1[cent2change][0].set_xlabel(f"Centroid {cent2change}")
    
       
## Draw how assignment changes and how it affecst loss function
for i in range(len(c_coord_range)):
    for cent2change in range(kmeans.num_centroids): 
        
        # get associated coordinates, loss and regions per centroid.
        _c,_loss,_cent_assigned = c_coord_range[i],loss_acc[cent2change][i],assigned_regions[cent2change][i]
        
        ax_list_1[cent2change][1].cla()

        # draw data assignment
        colors_data = [f"C{i}" for i in _cent_assigned]     
        ax_list_1[cent2change][1].scatter(X, np.zeros_like(X), marker = 'x',  c = colors_data)

        # highlight loss
        ax_list_1[cent2change][0].plot( _c, _loss, 'o' ,c = 'k')
        
        # remove previous highlighted
        if i != 0:
            ax_list_1[cent2change][0].plot(  c_coord_range[i-1], loss_acc[cent2change][i-1], 'o' , c = region_colors[cent2change][i-1])

        ## Get centroids from the algorithm
        centroids = kmeans.centroids

        # centroid
        centroids[cent2change,:] = _c

        ## draw centroid
        for idx,cet in enumerate(centroids):
            ## plot centroids
            ax_list_1[cent2change][1].plot(cet,0.0, 'o', color = f'C{idx}')

        ## draw voronoi regions
        voronoi_regions, _ = assign_data_to_centroids(vor_grid_xx , centroids)
        voronoi_regions_limits = np.where(np.diff(voronoi_regions) != 0)[0] + 1

        for c in voronoi_regions_limits:
            ax_list_1[cent2change][1].axvline(vor_grid_xx[c], color='k', linestyle="--")
            
        # fix xlim
        ax_list_1[cent2change][1].set_xlim([grid_lim_l, grid_lim_u])
        

    fig_1.canvas.draw()
    frame = np.asarray(fig_1.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)

writer.close()     

display(Video(data=video_filename, embed=True))
os.remove(video_filename)
plt.close(fig_1)

In [ ]:
N_grid = 100

grid_lim_l_c0 = -10
grid_lim_u_c0 = 10

grid_lim_l_c1 = -4
grid_lim_u_c1 = 4
cA_coord_mesh,cB_coord_mesh = np.meshgrid(np.linspace(grid_lim_l_c0,grid_lim_u_c0,N_grid),np.linspace(grid_lim_l_c1,grid_lim_u_c1,N_grid))
c_mesh = np.hstack((np.reshape(cA_coord_mesh, (N_grid**2,1)),np.reshape(cB_coord_mesh, (N_grid**2,1))))
c_mesh = c_mesh[:,:, np.newaxis]


fig = plt.figure(figsize=(20, 10))
ax1 = fig.add_subplot(1, 2, 1) 
ax2 = fig.add_subplot(1, 2, 2, projection='3d')  
ax2.view_init(elev=30, azim=-90)

ax_list = [ax1, ax2]

## Get parameters and/or variables which are fixed 
locally_convex = False
true_centroids_assigned, true_X_assigned  = kmeans._assign_data_to_centroids(X)


## ====================
## Parameter to vary ##
# both clusters at the same time
loss_acc = []
for centroids in c_mesh:
    
    # print(c_mesh)
    
    if locally_convex:
        centroids_assigned, X_assigned = true_centroids_assigned, true_X_assigned 
    else:
        centroids_assigned, X_assigned = kmeans._assign_data_to_centroids(X, centroids = centroids)

    loss = kmeans.loss_function(X, centroids = centroids, X_assigned = X_assigned)
    loss_acc.append(loss)

loss_acc_mesh = np.reshape(np.array(loss_acc), (N_grid,N_grid))

ax_list[0].contourf(cA_coord_mesh, cB_coord_mesh, loss_acc_mesh, cmap = plt.cm.get_cmap("Oranges_r"), levels = 30)
ax_list[1].plot_surface(cA_coord_mesh, cB_coord_mesh, loss_acc_mesh, cmap = plt.cm.get_cmap("Oranges_r"))

ax_list[0].set_xlabel("centroid 0")
ax_list[0].set_ylabel("centroid 1")

ax_list[1].set_xlabel("centroid 0")
ax_list[1].set_ylabel("centroid 1")

**3 clusters N = 8**

In [ ]:
interactive_plot = False

if interactive_plot:
    %matplotlib tk
else:
    %matplotlib inline
    
plotter = Plot1DKmeans(
                        video = True, 
                        interactive_plot = interactive_plot, 
                        sleep_time = 0.1, 
                        draw_voronoi = True,
                      )

kmeans = Kmeans(num_centroids = 3, plotter = plotter)
kmeans.initialize_centroids(X, seed = 1) 

In [ ]:
# video creation
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")  

# Plotting specifications
N_grid = 100
grid_lim_l = -10
grid_lim_u = 10
c_coord_range = np.reshape(np.linspace(grid_lim_l,grid_lim_u,N_grid),(N_grid,1))
vor_grid_xx = np.reshape( np.linspace(grid_lim_l, grid_lim_u, 500), (500,1))

fig_1, ax_list_1 = plt.subplots(kmeans.num_centroids,2, figsize = (10,5))

## Locally convex vs wthing assginment change
locally_convex = False
true_centroids_assigned, true_X_assigned = kmeans._assign_data_to_centroids(X)

## ====================
## Parameter to vary ##
## Centroid and coordinate

## to separate the cost function within the different assignments
assigned_regions = {}

# to keep loss
loss_acc = {}

# to keep region colors
region_colors = {}

for cent2change in range(kmeans.num_centroids):
    
    ## to separate the cost function within the different assignments
    assigned_regions[cent2change] = []

    ## Get centroids from the algorithm
    centroids = kmeans.centroids

    # to keep loss
    loss_acc[cent2change] = []
    
    for _c in c_coord_range:
        centroids[cent2change,:] = _c

        if locally_convex:
            centroids_assigned, X_assigned = true_centroids_assigned, true_X_assigned 
        else:
            centroids_assigned, X_assigned = kmeans._assign_data_to_centroids(X, centroids = centroids)

        assigned_regions[cent2change].append(tuple(centroids_assigned))

        loss = kmeans.loss_function(X, centroids = centroids,  X_assigned = X_assigned)
        loss_acc[cent2change].append(loss)


    unique_assignments = list(set(assigned_regions[cent2change]))
    region_colors[cent2change] = [f"C{unique_assignments.index(a)}" for a in assigned_regions[cent2change]]
   
    ## ==================
    ## Draw loss function
    ax_list_1[cent2change][0].plot(c_coord_range, loss_acc[cent2change], c = 'C0')
    ax_list_1[cent2change][0].scatter(c_coord_range, loss_acc[cent2change], c = region_colors[cent2change])

    ax_list_1[cent2change][0].set_xlabel(f"Centroid {cent2change}")
    
       
## Draw how assignment changes and how it affecst loss function
for i in range(0,len(c_coord_range),5):
    for cent2change in range(kmeans.num_centroids): 
        
        # get associated coordinates, loss and regions per centroid.
        _c,_loss,_cent_assigned = c_coord_range[i],loss_acc[cent2change][i],assigned_regions[cent2change][i]
        
        ax_list_1[cent2change][1].cla()

        # draw data assignment
        colors_data = [f"C{i}" for i in _cent_assigned]     
        ax_list_1[cent2change][1].scatter(X, np.zeros_like(X), marker = 'x',  c = colors_data)

        # highlight loss
        ax_list_1[cent2change][0].plot( _c, _loss, 'o' ,c = 'k')#, markersize =10)
        
        # remove previous highlighted
        if i != 0:
            ax_list_1[cent2change][0].plot(  c_coord_range[i-1], loss_acc[cent2change][i-1], 'o' , c = region_colors[cent2change][i-1])

        ## Get centroids from the algorithm
        centroids = kmeans.centroids

        # centroid
        centroids[cent2change,:] = _c

        ## draw centroid
        for idx,cet in enumerate(centroids):
            ## plot centroids
            ax_list_1[cent2change][1].plot(cet,0.0, 'o', color = f'C{idx}')

        ## draw voronoi regions
        voronoi_regions, _ = assign_data_to_centroids(vor_grid_xx , centroids)
        voronoi_regions_limits = np.where(np.diff(voronoi_regions) != 0)[0] + 1

        for c in voronoi_regions_limits:
            ax_list_1[cent2change][1].axvline(vor_grid_xx[c], color='k', linestyle="--")
            
        # fix xlim
        ax_list_1[cent2change][1].set_xlim([grid_lim_l, grid_lim_u])
        

    fig_1.canvas.draw()
    frame = np.asarray(fig_1.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)

writer.close()     

display(Video(data=video_filename, embed=True))
os.remove(video_filename)
plt.close(fig_1)

In [ ]:
# video creation
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")  

# Plotting specifications
N_grid = 100
grid_lim_l = -10
grid_lim_u = 10
c_coord_range = np.reshape(np.linspace(grid_lim_l,grid_lim_u,N_grid),(N_grid,1))
vor_grid_xx = np.reshape( np.linspace(grid_lim_l, grid_lim_u, 500), (500,1))

fig_1, ax_list_1 = plt.subplots(kmeans.num_centroids,2, figsize = (10,5))

## Locally convex vs wthing assginment change
locally_convex = True
true_centroids_assigned, true_X_assigned = kmeans._assign_data_to_centroids(X)

## ====================
## Parameter to vary ##
## Centroid and coordinate

## to separate the cost function within the different assignments
assigned_regions = {}

# to keep loss
loss_acc = {}

# to keep region colors
region_colors = {}

for cent2change in range(kmeans.num_centroids):
    
    ## to separate the cost function within the different assignments
    assigned_regions[cent2change] = []

    ## Get centroids from the algorithm
    centroids = kmeans.centroids

    # to keep loss
    loss_acc[cent2change] = []
    
    for _c in c_coord_range:
        centroids[cent2change,:] = _c

        if locally_convex:
            centroids_assigned, X_assigned = true_centroids_assigned, true_X_assigned 
        else:
            centroids_assigned, X_assigned = kmeans._assign_data_to_centroids(X, centroids = centroids)

        assigned_regions[cent2change].append(tuple(centroids_assigned))

        loss = kmeans.loss_function(X, centroids = centroids,  X_assigned = X_assigned)
        loss_acc[cent2change].append(loss)


    unique_assignments = list(set(assigned_regions[cent2change]))
    region_colors[cent2change] = [f"C{unique_assignments.index(a)}" for a in assigned_regions[cent2change]]
   
    ## ==================
    ## Draw loss function
    ax_list_1[cent2change][0].plot(c_coord_range, loss_acc[cent2change], c = 'C0')
    ax_list_1[cent2change][0].scatter(c_coord_range, loss_acc[cent2change], c = region_colors[cent2change])

    ax_list_1[cent2change][0].set_xlabel(f"Centroid {cent2change}")
    
       
## Draw how assignment changes and how it affecst loss function
for i in range(0,len(c_coord_range),5):
    for cent2change in range(kmeans.num_centroids): 
        
        # get associated coordinates, loss and regions per centroid.
        _c,_loss,_cent_assigned = c_coord_range[i],loss_acc[cent2change][i],assigned_regions[cent2change][i]
        
        ax_list_1[cent2change][1].cla()

        # draw data assignment
        colors_data = [f"C{i}" for i in _cent_assigned]     
        ax_list_1[cent2change][1].scatter(X, np.zeros_like(X), marker = 'x',  c = colors_data)

        # highlight loss
        ax_list_1[cent2change][0].plot( _c, _loss, 'o' ,c = 'k')#, markersize =10)
        
        # remove previous highlighted
        if i != 0:
            ax_list_1[cent2change][0].plot(  c_coord_range[i-1], loss_acc[cent2change][i-1], 'o' , c = region_colors[cent2change][i-1])

        ## Get centroids from the algorithm
        centroids = kmeans.centroids

        # centroid
        centroids[cent2change,:] = _c

        ## draw centroid
        for idx,cet in enumerate(centroids):
            ## plot centroids
            ax_list_1[cent2change][1].plot(cet,0.0, 'o', color = f'C{idx}')

        ## draw voronoi regions
        voronoi_regions, _ = assign_data_to_centroids(vor_grid_xx , centroids)
        voronoi_regions_limits = np.where(np.diff(voronoi_regions) != 0)[0] + 1

        for c in voronoi_regions_limits:
            ax_list_1[cent2change][1].axvline(vor_grid_xx[c], color='k', linestyle="--")
            
        # fix xlim
        ax_list_1[cent2change][1].set_xlim([grid_lim_l, grid_lim_u])
        

    fig_1.canvas.draw()
    frame = np.asarray(fig_1.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)

writer.close()     

display(Video(data=video_filename, embed=True))
os.remove(video_filename)
plt.close(fig_1)

##### Loss function for 2-dimensional data

Let's increase the dimensionality of the problem and consider data in $\mathbb{R}^2$. We can plot the marginal loss function at some coordinate for a given centroid, or the joint loss function considering different combinations of coordinates between centroids, or a centroid itself.

In [ ]:
np.random.seed(1)

N_points = 100

## probability p(c)
p = [0.33,0.33,0.34]
c = [0,1,2]

## probability p(x|c)
p_xc = {
    'c_0' : {
        'mu' : np.array([0,0]),
        'cov' : 0.05*np.eye(2),
        },
    'c_1' : {
        'mu' : np.array([0.5,0.5]),
        'cov' : 0.05*np.eye(2),
        },
    'c_2' : {
        'mu' : np.array([0,0.5]),
        'cov' : 0.05*np.eye(2),
        }   
}

## sample cluster assignments
cluster = np.random.choice(c, size=N_points, p=p)

## 
X_clust = {}
X = np.zeros((N_points,2), dtype = np.float32)
counter = 0
for _c in c:
    num_c = np.sum(cluster==_c)
    
    mu  = p_xc[f'c_{_c}']['mu']
    cov = p_xc[f'c_{_c}']['cov']
    
    _x = np.random.multivariate_normal(mean=mu, cov=cov, size = num_c)
    
    X[counter:counter + num_c] = _x
    X_clust[_c] = _x
    
    counter += num_c


fig, (ax1,ax2) = plt.subplots(1,2, figsize = (10,5))
ax1.plot(X[:,0],X[:,1], 'x', color = 'k')
ax1.set_xlabel(r"$x_1$")
ax1.set_ylabel(r"$x_2$")
ax1.set_title("Unlabelled Data")

for _c in c:
    _x = X_clust[_c]
    ax2.plot(_x[:,0],_x[:,1], 'x', color = f"C{_c}")
    ax2.set_xlabel(r"$x_1$")
    ax2.set_ylabel(r"$x_2$")
    ax2.set_title("True Cluster Assignment")

## shuffle data
for i in range(10):
    np.random.shuffle(X)

Then I will run the kmean algorithm:

In [ ]:
interactive_plot = False

if interactive_plot:
    %matplotlib tk
    plt.close("all")
else:
    %matplotlib inline
    
plotter = Plot2DKmeans(
                        video = True, 
                        interactive_plot = interactive_plot, 
                        sleep_time = 0.01, 
                        draw_voronoi = True,
                        # loss_fun_optim = True,
                        # num_loss_fun_centroids_to_show = 3 
                      )

kmeans = Kmeans(num_centroids = 4, plotter = plotter)
kmeans.run(X, num_iters = 20, seed = 1)

plotter.show_video()

I will fix $c_n$ to the assignments obtained after fitting the algorithm using the optimal centroids. With this fixed value for $c_n$ is time to see loss function varies with centroid values. I will vary the values of different parts of $\muvec_k$, fixing the others to the ones obtained after running the algorithm.

**1d Loss function**

First, I will show the loss function fixing all parameters except a single coordinate of each centroid.

In [ ]:
# Plotting specifications
N_grid = 100
grid_lim_l = -2
grid_lim_u = 2
c_coord_range = np.reshape(np.linspace(grid_lim_l,grid_lim_u,N_grid),(N_grid,1))

fig, ax_list = plt.subplots(kmeans.num_centroids,2, figsize = (20,20))

## Locally convex vs wthing assginment change
locally_convex = False
true_centroids_assigned, true_X_assigned = kmeans._assign_data_to_centroids(X)


## ====================
## Parameter to vary ##
## Centroid and coordinate

for cent2change in range(kmeans.num_centroids):
    for coord2change in range(2):
        
        ## to separate the cost function within the different assignments
        assigned_regions = []
        
        ## Get centroids from the algorithm
        centroids = kmeans.centroids

        if coord2change == 1:
            c_range = np.hstack((centroids[cent2change,0]*np.ones((N_grid,1)),c_coord_range))    

        elif coord2change == 0:
            c_range = np.hstack((c_coord_range,centroids[cent2change,1]*np.ones((N_grid,1))))

        else:
            raise ValueError("")

        loss_acc = []
        for _c in c_range:
            centroids[cent2change,:] = _c
            
            if locally_convex:
                centroids_assigned, X_assigned = true_centroids_assigned, true_X_assigned 
            else:
                centroids_assigned, X_assigned = kmeans._assign_data_to_centroids(X, centroids = centroids)
            
            assigned_regions.append(tuple(centroids_assigned))
            
            loss = kmeans.loss_function(X, centroids = centroids, X_assigned = X_assigned)
            loss_acc.append(loss)

    
        unique_assignments = list(set(assigned_regions))
        region_colors = [unique_assignments.index(a) for a in assigned_regions]
        
        region_colors = np.array(region_colors)
        region_colors = (region_colors - region_colors.min()) / (region_colors.max() - region_colors.min())
        
        colormap = plt.cm.get_cmap("Oranges") 
        region_colors = colormap(region_colors)
        #region_colors = [tuple(c) for c in region_colors]
        
        ax_list[cent2change,coord2change].plot(c_coord_range, loss_acc, c = 'C0')
        ax_list[cent2change,coord2change].scatter(c_coord_range, loss_acc, c = region_colors)
        
        ax_list[cent2change,coord2change].xaxis.set_major_locator(MultipleLocator(0.2))
        ax_list[cent2change,coord2change].xaxis.set_major_formatter(FormatStrFormatter('%.1f'))

        if coord2change == 0:
            ax_list[cent2change,coord2change].set_ylabel(f"Centroid {cent2change}")
            
        if cent2change == kmeans.num_centroids-1:
            ax_list[cent2change,coord2change].set_xlabel(f"Coordinate {coord2change}")

In [ ]:
# Plotting specifications
N_grid = 100
grid_lim_l = -2
grid_lim_u = 2
c_coord_range = np.reshape(np.linspace(grid_lim_l,grid_lim_u,N_grid),(N_grid,1))

fig, ax_list = plt.subplots(kmeans.num_centroids,2, figsize = (20,20))

## Locally convex vs wthing assginment change
locally_convex = True
true_centroids_assigned, true_X_assigned = kmeans._assign_data_to_centroids(X)


## ====================
## Parameter to vary ##
## Centroid and coordinate

for cent2change in range(kmeans.num_centroids):
    for coord2change in range(2):
        
        ## to separate the cost function within the different assignments
        assigned_regions = []
        
        ## Get centroids from the algorithm
        centroids = kmeans.centroids

        if coord2change == 1:
            c_range = np.hstack((centroids[cent2change,0]*np.ones((N_grid,1)),c_coord_range))    

        elif coord2change == 0:
            c_range = np.hstack((c_coord_range,centroids[cent2change,1]*np.ones((N_grid,1))))

        else:
            raise ValueError("")

        loss_acc = []
        for _c in c_range:
            centroids[cent2change,:] = _c
            
            if locally_convex:
                centroids_assigned, X_assigned = true_centroids_assigned, true_X_assigned 
            else:
                centroids_assigned, X_assigned = kmeans._assign_data_to_centroids(X, centroids = centroids)
            
            assigned_regions.append(tuple(centroids_assigned))
            
            loss = kmeans.loss_function(X, centroids = centroids, X_assigned = X_assigned)
            loss_acc.append(loss)

    
        unique_assignments = list(set(assigned_regions))
        region_colors = [unique_assignments.index(a) for a in assigned_regions]
        
        region_colors = np.array(region_colors)
        region_colors = (region_colors - region_colors.min()) / (region_colors.max() - region_colors.min())
        
        colormap = plt.cm.get_cmap("Oranges") 
        region_colors = colormap(region_colors)
        #region_colors = [tuple(c) for c in region_colors]
        
        ax_list[cent2change,coord2change].plot(c_coord_range, loss_acc, c = 'C0')
        ax_list[cent2change,coord2change].scatter(c_coord_range, loss_acc, c = region_colors)
        
        ax_list[cent2change,coord2change].xaxis.set_major_locator(MultipleLocator(0.2))
        ax_list[cent2change,coord2change].xaxis.set_major_formatter(FormatStrFormatter('%.1f'))

        if coord2change == 0:
            ax_list[cent2change,coord2change].set_ylabel(f"Centroid {cent2change}")
            
        if cent2change == kmeans.num_centroids-1:
            ax_list[cent2change,coord2change].set_xlabel(f"Coordinate {coord2change}")

**2d Loss function showing a single centroid**

Time for the loss function varying each centroid along both coordinates.

In [ ]:
print(kmeans.centroids)

N_grid = 100
grid_lim_l = -1.2
grid_lim_u = 1.2

cA_coord_mesh,cB_coord_mesh = np.meshgrid(np.linspace(grid_lim_l,grid_lim_u,N_grid),np.linspace(grid_lim_l,grid_lim_u,N_grid))
c_mesh = np.hstack((np.reshape(cA_coord_mesh, (N_grid**2,1)),np.reshape(cB_coord_mesh, (N_grid**2,1))))


fig = plt.figure(figsize=(10, 20))  
gs = gridspec.GridSpec(4, 2, width_ratios=[2.0, 1])  

ax_list = []

for i in range(4):
    ax1 = fig.add_subplot(gs[i, 0], projection='3d')  
    ax1.view_init(elev=50, azim=20)

    ax2 = fig.add_subplot(gs[i, 1])  

    ax_list.append([ax1, ax2])
    
## Get parameters and/or variables which are fixed 
locally_convex = False
true_centroids_assigned, true_X_assigned  = kmeans._assign_data_to_centroids(X)

## ====================
## Parameter to vary ##
## There are eight combination
for cent2change in range(4):
    
    ## Get centroids from the algorithm
    centroids = copy.deepcopy(kmeans.centroids)

    loss_acc = []
    for _c in c_mesh:
        centroids[cent2change,:] = _c

        if locally_convex:
            centroids_assigned, X_assigned = true_centroids_assigned, true_X_assigned 
        else:
            centroids_assigned, X_assigned = kmeans._assign_data_to_centroids(X, centroids = centroids)
        
        loss = kmeans.loss_function(X, centroids = centroids, X_assigned = X_assigned)
        loss_acc.append(loss)

    loss_acc_mesh = np.reshape(np.array(loss_acc), (N_grid,N_grid))

    ax_list[cent2change][0].plot_surface(cA_coord_mesh, cB_coord_mesh, loss_acc_mesh, cmap = plt.cm.get_cmap("Oranges"))
    ax_list[cent2change][1].contourf(cA_coord_mesh, cB_coord_mesh, loss_acc_mesh, cmap = plt.cm.get_cmap("Oranges"), levels = 30)

    ax_list[cent2change][0].set_title(f"Centroid {cent2change}")
    
    ax_list[cent2change][0].set_xlabel(f"Coordinate {0}")
    ax_list[cent2change][0].set_ylabel(f"Coordinate {1}")
    ax_list[cent2change][1].set_xlabel(f"Coordinate {0}")
    ax_list[cent2change][1].set_ylabel(f"Coordinate {1}")

In [ ]:
N_grid = 100
grid_lim_l = -1.2
grid_lim_u = 1.2

cA_coord_mesh,cB_coord_mesh = np.meshgrid(np.linspace(grid_lim_l,grid_lim_u,N_grid),np.linspace(grid_lim_l,grid_lim_u,N_grid))
c_mesh = np.hstack((np.reshape(cA_coord_mesh, (N_grid**2,1)),np.reshape(cB_coord_mesh, (N_grid**2,1))))


fig = plt.figure(figsize=(10, 20))  
gs = gridspec.GridSpec(4, 2, width_ratios=[2.0, 1])  

ax_list = []

for i in range(4):
    ax1 = fig.add_subplot(gs[i, 0], projection='3d')  
    ax1.view_init(elev=50, azim=20)

    ax2 = fig.add_subplot(gs[i, 1])  

    ax_list.append([ax1, ax2])
    
## Get parameters and/or variables which are fixed 
locally_convex = True
true_centroids_assigned, true_X_assigned  = kmeans._assign_data_to_centroids(X)

## ====================
## Parameter to vary ##
## There are eight combination
for cent2change in range(4):
    
    ## Get centroids from the algorithm
    centroids = kmeans.centroids

    loss_acc = []
    for _c in c_mesh:
        centroids[cent2change,:] = _c

        if locally_convex:
            centroids_assigned, X_assigned = true_centroids_assigned, true_X_assigned 
        else:
            centroids_assigned, X_assigned = kmeans._assign_data_to_centroids(X, centroids = centroids)
        
        loss = kmeans.loss_function(X, centroids = centroids, X_assigned = X_assigned)
        loss_acc.append(loss)

    loss_acc_mesh = np.reshape(np.array(loss_acc), (N_grid,N_grid))

    ax_list[cent2change][0].plot_surface(cA_coord_mesh, cB_coord_mesh, loss_acc_mesh, cmap = plt.cm.get_cmap("Oranges"))
    ax_list[cent2change][1].contourf(cA_coord_mesh, cB_coord_mesh, loss_acc_mesh, cmap = plt.cm.get_cmap("Oranges"), levels = 30)

    ax_list[cent2change][0].set_title(f"Centroid {cent2change}")
    
    ax_list[cent2change][0].set_xlabel(f"Coordinate {0}")
    ax_list[cent2change][0].set_ylabel(f"Coordinate {1}")
    ax_list[cent2change][1].set_xlabel(f"Coordinate {0}")
    ax_list[cent2change][1].set_ylabel(f"Coordinate {1}")

**2d Loss function comparing centroids' coordinates**

Time for the loss function while varying the coordinates of different centroids.

In [ ]:
N_grid = 100
grid_lim_l = -1.4
grid_lim_u = 2

cA_coord_range,cB_coord_range = np.linspace(grid_lim_l,grid_lim_u,N_grid), np.linspace(grid_lim_l,grid_lim_u,N_grid)
cA_coord_mesh, cB_coord_mesh = np.meshgrid(cA_coord_range,cB_coord_range)

combinations = [
    [(0,0),(1,0)], # centroid 0 coordinate 0 vs centroid 1 coordinate 0
    [(0,0),(1,1)],
    [(0,1),(1,0)],
    [(0,1),(1,1)],
    #
    [(1,0),(2,0)], 
    [(1,0),(2,1)],
    [(1,1),(2,0)],
    [(1,1),(2,1)],
    #
    [(2,0),(3,0)], 
    [(2,0),(3,1)],
    [(2,1),(3,0)],
    [(2,1),(3,1)],
    #
]

fig = plt.figure(figsize=(30, 120))  
gs = gridspec.GridSpec(len(combinations), 2, width_ratios=[2, 1])  

ax_list = []
for i in range(len(combinations)):
    ax1 = fig.add_subplot(gs[i, 0], projection='3d')  
    ax1.view_init(elev=70, azim=90)

    ax2 = fig.add_subplot(gs[i, 1])  

    ax_list.append([ax1, ax2])


## Get parameters and/or variables which are fixed 
locally_convex = False
true_centroids_assigned, true_X_assigned = kmeans._assign_data_to_centroids(X)


for idx, comb in enumerate(combinations):
    (c0,coord0),(c1,coord1) = comb
    
    ## Get centroids from the algorithm
    centroids = kmeans.centroids

    ## ====================
    ## Parameter to vary ##
    ## There are eight combination
    loss_acc_mesh = np.zeros((N_grid, N_grid))
    for r,_cB in enumerate(cB_coord_range):
        for c,_cA in enumerate(cA_coord_range):

            centroids[c0,coord0] = _cA
            centroids[c1,coord1] = _cB
            
            if locally_convex:
                centroids_assigned, X_assigned = true_centroids_assigned, true_X_assigned 
            else:
                centroids_assigned, X_assigned = kmeans._assign_data_to_centroids(X, centroids = centroids)

            loss = kmeans.loss_function(X, centroids = centroids, X_assigned = X_assigned)
            loss_acc_mesh[r,c] = loss

    ax_list[idx][0].plot_surface(cA_coord_mesh, cB_coord_mesh, loss_acc_mesh, cmap = plt.cm.get_cmap("Oranges_r"))
    ax_list[idx][1].contourf(cA_coord_mesh, cB_coord_mesh, loss_acc_mesh, cmap = plt.cm.get_cmap("Oranges_r"), levels = 30)
    
    ax_list[idx][0].set_xlabel(f"Centroid {c0} Coordinate {coord0}")
    ax_list[idx][0].set_ylabel(f"Centroid {c1} Coordinate {coord1}")
    ax_list[idx][1].set_xlabel(f"Centroid {c0} Coordinate {coord0}")
    ax_list[idx][1].set_ylabel(f"Centroid {c1} Coordinate {coord1}")

In [ ]:
N_grid = 100
grid_lim_l = -1.4
grid_lim_u = 2

cA_coord_range,cB_coord_range = np.linspace(grid_lim_l,grid_lim_u,N_grid), np.linspace(grid_lim_l,grid_lim_u,N_grid)
cA_coord_mesh, cB_coord_mesh = np.meshgrid(cA_coord_range,cB_coord_range)

combinations = [
    [(0,0),(1,0)], # centroid 0 coordinate 0 vs centroid 1 coordinate 0
    [(0,0),(1,1)],
    [(0,1),(1,0)],
    [(0,1),(1,1)],
    #
    [(1,0),(2,0)], 
    [(1,0),(2,1)],
    [(1,1),(2,0)],
    [(1,1),(2,1)],
    #
    [(2,0),(3,0)], 
    [(2,0),(3,1)],
    [(2,1),(3,0)],
    [(2,1),(3,1)],
    #
]

fig = plt.figure(figsize=(30, 120))  
gs = gridspec.GridSpec(len(combinations), 2, width_ratios=[2, 1])  

ax_list = []
for i in range(len(combinations)):
    ax1 = fig.add_subplot(gs[i, 0], projection='3d')  
    ax1.view_init(elev=70, azim=90)

    ax2 = fig.add_subplot(gs[i, 1])  

    ax_list.append([ax1, ax2])


## Get parameters and/or variables which are fixed 
locally_convex = True
true_centroids_assigned, true_X_assigned = kmeans._assign_data_to_centroids(X)


for idx, comb in enumerate(combinations):
    (c0,coord0),(c1,coord1) = comb
    
    ## Get centroids from the algorithm
    centroids = kmeans.centroids

    ## ====================
    ## Parameter to vary ##
    ## There are eight combination
    loss_acc_mesh = np.zeros((N_grid, N_grid))
    for r,_cB in enumerate(cB_coord_range):
        for c,_cA in enumerate(cA_coord_range):

            centroids[c0,coord0] = _cA
            centroids[c1,coord1] = _cB
            
            if locally_convex:
                centroids_assigned, X_assigned = true_centroids_assigned, true_X_assigned 
            else:
                centroids_assigned, X_assigned = kmeans._assign_data_to_centroids(X, centroids = centroids)

            loss = kmeans.loss_function(X, centroids = centroids, X_assigned = X_assigned)
            loss_acc_mesh[r,c] = loss

    ax_list[idx][0].plot_surface(cA_coord_mesh, cB_coord_mesh, loss_acc_mesh, cmap = plt.cm.get_cmap("Oranges_r"))
    ax_list[idx][1].contourf(cA_coord_mesh, cB_coord_mesh, loss_acc_mesh, cmap = plt.cm.get_cmap("Oranges_r"), levels = 30)
    
    ax_list[idx][0].set_xlabel(f"Centroid {c0} Coordinate {coord0}")
    ax_list[idx][0].set_ylabel(f"Centroid {c1} Coordinate {coord1}")
    ax_list[idx][1].set_xlabel(f"Centroid {c0} Coordinate {coord0}")
    ax_list[idx][1].set_ylabel(f"Centroid {c1} Coordinate {coord1}")

### Coordinate descent algorithm optimization: optimal steps

Let's now visualize how the coordinate descent algorithm works. There are two sets of variables over which we optimize: the cluster assignments $c_n$ and cluster centroid values $\muvec_k$. Coordinate descent alternates between fixing $\muvec_k$ and getting optimal $c_n$ (step 2 in the algorithm) and fixing $c_n$ and getting optimal $\muvec_k$ (step 3 in the algorithm). Within each coordinate step, a group of variables is being optimized. For example step 3 get optimal values for all $\muvec_k$ at the same time because optimallity of each $\muvec_k$ is independent of other optimal value $\muvec_{k'}$

#### 1 dimensional data

##### 2 clusters N = 3 points
Here is the data used.

In [ ]:
np.random.seed(1)

N_points = 3

## probability p(c)
p = [0.33,0.33,0.34]
c = [0,1,2]

## probability p(x|c)
var1 = 0.1
var2 = 0.01
var3 = 0.01
p_xc = {
    'c_0' : {
        'mu' : np.array([0]),
        'cov' : var1*np.eye(1),
        },
    'c_1' : {
        'mu' : np.array([3]),
        'cov' : var2*np.eye(1),
        },
    'c_2' : {
        'mu' : np.array([5]),
        'cov' : var3*np.eye(1),
        }   
}

## sample cluster assignments
cluster = np.random.choice(c, size=N_points, p=p)

## 
X = np.zeros((N_points,1), dtype = np.float32)
counter = 0
X_clust = {}
for _c in c:
    num_c = np.sum(cluster==_c)
    
    mu  = p_xc[f'c_{_c}']['mu']
    cov = p_xc[f'c_{_c}']['cov']
    
    _x = np.random.multivariate_normal(mean=mu, cov=cov, size = num_c)
    
    X[counter:counter + num_c] = _x
    X_clust[_c] = _x
    
    counter += num_c

fig, (ax1,ax2) = plt.subplots(1,2, figsize = (10,5))
ax1.plot(X[:,0],np.zeros_like(X), 'x', color = 'k')
ax1.set_xlabel(r"$x_1$")
ax1.set_title("Unlabelled Data")

for _c in c:
    _x = X_clust[_c]
    ax2.plot(_x[:,0],np.zeros_like(_x), 'x', color = f"C{_c}")
    ax2.set_xlabel(r"$x_1$")
    ax2.set_title("True Cluster Assignment")


## shuffle data
for i in range(10):
    np.random.shuffle(X)

Initial cluster assignments

In [ ]:
interactive_plot = False

if interactive_plot:
    %matplotlib tk
    plt.close("all")
else:
    %matplotlib inline
    
plotter = Plot1DKmeans(
                        video = True, 
                        interactive_plot = interactive_plot, 
                        sleep_time = 0.01, 
                        draw_voronoi = True,
                        loss_fun_optim = False,
                        num_loss_fun_centroids_to_show = 2
                      )

kmeans = Kmeans(num_centroids = 2, plotter = plotter)
kmeans.initialize_centroids(X, seed = 1) 

Loss function given initial centroids.

In [ ]:
# video creation
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")  

# Plotting specifications
N_grid = 100
grid_lim_l = -10
grid_lim_u = 10
c_coord_range = np.reshape(np.linspace(grid_lim_l,grid_lim_u,N_grid),(N_grid,1))
vor_grid_xx = np.reshape( np.linspace(grid_lim_l, grid_lim_u, 500), (500,1))

fig_1, ax_list_1 = plt.subplots(kmeans.num_centroids,2, figsize = (10,5))

## Locally convex vs wthing assginment change
locally_convex = False
true_centroids_assigned, true_X_assigned = kmeans._assign_data_to_centroids(X)

## ====================
## Parameter to vary ##
## Centroid and coordinate

## to separate the cost function within the different assignments
assigned_regions = {}

# to keep loss
loss_acc = {}

# to keep region colors
region_colors = {}

for cent2change in range(kmeans.num_centroids):
    
    ## to separate the cost function within the different assignments
    assigned_regions[cent2change] = []

    ## Get centroids from the algorithm
    centroids = kmeans.centroids

    # to keep loss
    loss_acc[cent2change] = []
    
    for _c in c_coord_range:
        centroids[cent2change,:] = _c

        if locally_convex:
            centroids_assigned, X_assigned = true_centroids_assigned, true_X_assigned 
        else:
            centroids_assigned, X_assigned = kmeans._assign_data_to_centroids(X, centroids = centroids)

        assigned_regions[cent2change].append(tuple(centroids_assigned))

        loss = kmeans.loss_function(X, centroids = centroids, X_assigned = X_assigned)
        loss_acc[cent2change].append(loss)


    unique_assignments = list(set(assigned_regions[cent2change]))
    region_colors[cent2change] = [f"C{unique_assignments.index(a)}" for a in assigned_regions[cent2change]]
   
    ## ==================
    ## Draw loss function
    ax_list_1[cent2change][0].plot(c_coord_range, loss_acc[cent2change], c = 'C0')
    ax_list_1[cent2change][0].scatter(c_coord_range, loss_acc[cent2change], c = region_colors[cent2change])

    ax_list_1[cent2change][0].set_xlabel(f"Centroid {cent2change}")
    
       
## Draw how assignment changes and how it affecst loss function
for i in range(len(c_coord_range)):
    for cent2change in range(kmeans.num_centroids): 
        
        # get associated coordinates, loss and regions per centroid.
        _c,_loss,_cent_assigned = c_coord_range[i],loss_acc[cent2change][i],assigned_regions[cent2change][i]
        
        ax_list_1[cent2change][1].cla()

        # draw data assignment
        colors_data = [f"C{i}" for i in _cent_assigned]     
        ax_list_1[cent2change][1].scatter(X, np.zeros_like(X), marker = 'x',  c = colors_data)

        # highlight loss
        ax_list_1[cent2change][0].plot( _c, _loss, 'o' ,c = 'k')#, markersize =10)
        
        # remove previous highlighted
        if i != 0:
            ax_list_1[cent2change][0].plot(  c_coord_range[i-1], loss_acc[cent2change][i-1], 'o' , c = region_colors[cent2change][i-1])

        ## Get centroids from the algorithm
        centroids = kmeans.centroids

        # centroid
        centroids[cent2change,:] = _c

        ## draw centroid
        for idx,cet in enumerate(centroids):
            ## plot centroids
            ax_list_1[cent2change][1].plot(cet,0.0, 'o', color = f'C{idx}')

        ## draw voronoi regions
        voronoi_regions, _ = assign_data_to_centroids(vor_grid_xx , centroids)
        voronoi_regions_limits = np.where(np.diff(voronoi_regions) != 0)[0] + 1

        for c in voronoi_regions_limits:
            ax_list_1[cent2change][1].axvline(vor_grid_xx[c], color='k', linestyle="--")
            
        # fix xlim
        ax_list_1[cent2change][1].set_xlim([grid_lim_l, grid_lim_u])
        

    fig_1.canvas.draw()
    frame = np.asarray(fig_1.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)

writer.close()     

display(Video(data=video_filename, embed=True))
os.remove(video_filename)
plt.close(fig_1)

Coordinate descent algorithm

In [ ]:
plotter = Plot1DKmeans(
                        video = True, 
                        interactive_plot = interactive_plot, 
                        sleep_time = 0.01, 
                        draw_voronoi = True,
                        loss_fun_optim = True,
                        num_loss_fun_centroids_to_show = 2
                      )

kmeans = Kmeans(num_centroids = 2, plotter = plotter)
kmeans.initialize_centroids(X, seed = 1) 

kmeans.run(X, num_iters = 10, seed = 1)

plotter.show_video()

##### 2 clusters N = 5 points

In [ ]:
np.random.seed(1)

N_points = 5

## probability p(c)
p = [0.33,0.33,0.34]
c = [0,1,2]

## probability p(x|c)
var1 = 0.1
var2 = 0.01
var3 = 0.01
p_xc = {
    'c_0' : {
        'mu' : np.array([0]),
        'cov' : var1*np.eye(1),
        },
    'c_1' : {
        'mu' : np.array([3]),
        'cov' : var2*np.eye(1),
        },
    'c_2' : {
        'mu' : np.array([5]),
        'cov' : var3*np.eye(1),
        }   
}

## sample cluster assignments
cluster = np.random.choice(c, size=N_points, p=p)

## 
X = np.zeros((N_points,1), dtype = np.float32)
counter = 0
X_clust = {}
for _c in c:
    num_c = np.sum(cluster==_c)
    
    mu  = p_xc[f'c_{_c}']['mu']
    cov = p_xc[f'c_{_c}']['cov']
    
    _x = np.random.multivariate_normal(mean=mu, cov=cov, size = num_c)
    
    X[counter:counter + num_c] = _x
    X_clust[_c] = _x
    
    counter += num_c

fig, (ax1,ax2) = plt.subplots(1,2, figsize = (10,5))
ax1.plot(X[:,0],np.zeros_like(X), 'x', color = 'k')
ax1.set_xlabel(r"$x_1$")
ax1.set_title("Unlabelled Data")

for _c in c:
    _x = X_clust[_c]
    ax2.plot(_x[:,0],np.zeros_like(_x), 'x', color = f"C{_c}")
    ax2.set_xlabel(r"$x_1$")
    ax2.set_title("True Cluster Assignment")


## shuffle data
for i in range(10):
    np.random.shuffle(X)

In [ ]:
interactive_plot = False

if interactive_plot:
    %matplotlib tk
    plt.close("all")
else:
    %matplotlib inline
    
plotter = Plot1DKmeans(
                        video = True, 
                        interactive_plot = interactive_plot, 
                        sleep_time = 0.01, 
                        draw_voronoi = True,
                        loss_fun_optim = False,
                        num_loss_fun_centroids_to_show = 2
                      )

kmeans = Kmeans(num_centroids = 2, plotter = plotter)
kmeans.initialize_centroids(X, seed = 1) 

In [ ]:
# video creation
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")  

# Plotting specifications
N_grid = 100
grid_lim_l = -10
grid_lim_u = 10
c_coord_range = np.reshape(np.linspace(grid_lim_l,grid_lim_u,N_grid),(N_grid,1))
vor_grid_xx = np.reshape( np.linspace(grid_lim_l, grid_lim_u, 500), (500,1))

fig_1, ax_list_1 = plt.subplots(kmeans.num_centroids,2, figsize = (10,5))

## Locally convex vs wthing assginment change
locally_convex = False
true_centroids_assigned, true_X_assigned = kmeans._assign_data_to_centroids(X)

## ====================
## Parameter to vary ##
## Centroid and coordinate

## to separate the cost function within the different assignments
assigned_regions = {}

# to keep loss
loss_acc = {}

# to keep region colors
region_colors = {}

for cent2change in range(kmeans.num_centroids):
    
    ## to separate the cost function within the different assignments
    assigned_regions[cent2change] = []

    ## Get centroids from the algorithm
    centroids = kmeans.centroids

    # to keep loss
    loss_acc[cent2change] = []
    
    for _c in c_coord_range:
        centroids[cent2change,:] = _c

        if locally_convex:
            centroids_assigned, X_assigned = true_centroids_assigned, true_X_assigned 
        else:
            centroids_assigned, X_assigned = kmeans._assign_data_to_centroids(X, centroids = centroids)

        assigned_regions[cent2change].append(tuple(centroids_assigned))

        loss = kmeans.loss_function(X, centroids = centroids, X_assigned = X_assigned)
        loss_acc[cent2change].append(loss)


    unique_assignments = list(set(assigned_regions[cent2change]))
    region_colors[cent2change] = [f"C{unique_assignments.index(a)}" for a in assigned_regions[cent2change]]
   
    ## ==================
    ## Draw loss function
    ax_list_1[cent2change][0].plot(c_coord_range, loss_acc[cent2change], c = 'C0')
    ax_list_1[cent2change][0].scatter(c_coord_range, loss_acc[cent2change], c = region_colors[cent2change])

    ax_list_1[cent2change][0].set_xlabel(f"Centroid {cent2change}")
    
       
## Draw how assignment changes and how it affecst loss function
for i in range(len(c_coord_range)):
    for cent2change in range(kmeans.num_centroids): 
        
        # get associated coordinates, loss and regions per centroid.
        _c,_loss,_cent_assigned = c_coord_range[i],loss_acc[cent2change][i],assigned_regions[cent2change][i]
        
        ax_list_1[cent2change][1].cla()

        # draw data assignment
        colors_data = [f"C{i}" for i in _cent_assigned]     
        ax_list_1[cent2change][1].scatter(X, np.zeros_like(X), marker = 'x',  c = colors_data)

        # highlight loss
        ax_list_1[cent2change][0].plot( _c, _loss, 'o' ,c = 'k')#, markersize =10)
        
        # remove previous highlighted
        if i != 0:
            ax_list_1[cent2change][0].plot(  c_coord_range[i-1], loss_acc[cent2change][i-1], 'o' , c = region_colors[cent2change][i-1])

        ## Get centroids from the algorithm
        centroids = kmeans.centroids

        # centroid
        centroids[cent2change,:] = _c

        ## draw centroid
        for idx,cet in enumerate(centroids):
            ## plot centroids
            ax_list_1[cent2change][1].plot(cet,0.0, 'o', color = f'C{idx}')

        ## draw voronoi regions
        voronoi_regions, _ = assign_data_to_centroids(vor_grid_xx , centroids)
        voronoi_regions_limits = np.where(np.diff(voronoi_regions) != 0)[0] + 1

        for c in voronoi_regions_limits:
            ax_list_1[cent2change][1].axvline(vor_grid_xx[c], color='k', linestyle="--")
            
        # fix xlim
        ax_list_1[cent2change][1].set_xlim([grid_lim_l, grid_lim_u])
        

    fig_1.canvas.draw()
    frame = np.asarray(fig_1.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)

writer.close()     

display(Video(data=video_filename, embed=True))
os.remove(video_filename)
plt.close(fig_1)

In [ ]:
plotter = Plot1DKmeans(
                        video = True, 
                        interactive_plot = interactive_plot, 
                        sleep_time = 0.01, 
                        draw_voronoi = True,
                        loss_fun_optim = True,
                        num_loss_fun_centroids_to_show = 2
                      )

kmeans = Kmeans(num_centroids = 2, plotter = plotter)
kmeans.run(X, num_iters = 10, seed = 1) 

plotter.show_video()

**Change initialization**

In [ ]:
plotter = Plot1DKmeans(
                        video = True, 
                        interactive_plot = interactive_plot, 
                        sleep_time = 0.01, 
                        draw_voronoi = True,
                        loss_fun_optim = True,
                        num_loss_fun_centroids_to_show = 2
                      )

kmeans = Kmeans(num_centroids = 2, plotter = plotter)
kmeans.run(X, num_iters = 10, seed = 4) 

plotter.show_video()

##### 3 clusters N = 50 points

If we recover our initial dataset, which is far more complex, we see that the complexity and the number of solutions we can obtain increase. Here, initialization plays a role.

In [ ]:
np.random.seed(1)

N_points = 50

## probability p(c)
p = [0.33,0.33,0.34]
c = [0,1,2]

## probability p(x|c)
var1 = 0.1
var2 = 0.01
var3 = 0.01
p_xc = {
    'c_0' : {
        'mu' : np.array([0]),
        'cov' : var1*np.eye(1),
        },
    'c_1' : {
        'mu' : np.array([0.2]),
        'cov' : var2*np.eye(1),
        },
    'c_2' : {
        'mu' : np.array([0.35]),
        'cov' : var3*np.eye(1),
        }   
}

## sample cluster assignments
cluster = np.random.choice(c, size=N_points, p=p)

## 
X = np.zeros((N_points,1), dtype = np.float32)
counter = 0
X_clust = {}
for _c in c:
    num_c = np.sum(cluster==_c)
    
    mu  = p_xc[f'c_{_c}']['mu']
    cov = p_xc[f'c_{_c}']['cov']
    
    _x = np.random.multivariate_normal(mean=mu, cov=cov, size = num_c)
    
    X[counter:counter + num_c] = _x
    X_clust[_c] = _x
    
    counter += num_c

fig, (ax1,ax2) = plt.subplots(1,2, figsize = (10,5))
ax1.plot(X[:,0],np.zeros_like(X), 'x', color = 'k')
ax1.set_xlabel(r"$x_1$")
ax1.set_ylabel(r"$x_2$")
ax1.set_title("Unlabelled Data")

for _c in c:
    _x = X_clust[_c]
    ax2.plot(_x[:,0],np.zeros_like(_x), 'x', color = f"C{_c}")
    ax2.set_xlabel(r"$x_1$")
    ax2.set_ylabel(r"$x_2$")
    ax2.set_title("True Cluster Assignment")


## shuffle data
for i in range(10):
    np.random.shuffle(X)

In [ ]:
# video creation
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")  

# Plotting specifications
N_grid = 100
grid_lim_l = -1
grid_lim_u = 1
c_coord_range = np.reshape(np.linspace(grid_lim_l,grid_lim_u,N_grid),(N_grid,1))
vor_grid_xx = np.reshape( np.linspace(grid_lim_l, grid_lim_u, 500), (500,1))

fig_1, ax_list_1 = plt.subplots(kmeans.num_centroids,2, figsize = (10,5))

## Locally convex vs wthing assginment change
locally_convex = False
true_centroids_assigned, true_X_assigned = kmeans._assign_data_to_centroids(X)

## ====================
## Parameter to vary ##
## Centroid and coordinate

## to separate the cost function within the different assignments
assigned_regions = {}

# to keep loss
loss_acc = {}

# to keep region colors
region_colors = {}

for cent2change in range(kmeans.num_centroids):
    
    ## to separate the cost function within the different assignments
    assigned_regions[cent2change] = []

    ## Get centroids from the algorithm
    centroids = kmeans.centroids

    # to keep loss
    loss_acc[cent2change] = []
    
    for _c in c_coord_range:
        centroids[cent2change,:] = _c

        if locally_convex:
            centroids_assigned, X_assigned = true_centroids_assigned, true_X_assigned 
        else:
            centroids_assigned, X_assigned = kmeans._assign_data_to_centroids(X, centroids = centroids)

        assigned_regions[cent2change].append(tuple(centroids_assigned))

        loss = kmeans.loss_function(X, centroids = centroids, X_assigned = X_assigned)
        loss_acc[cent2change].append(loss)


    unique_assignments = list(set(assigned_regions[cent2change]))
    region_colors[cent2change] = [f"C{unique_assignments.index(a)}" for a in assigned_regions[cent2change]]
   
    ## ==================
    ## Draw loss function
    ax_list_1[cent2change][0].plot(c_coord_range, loss_acc[cent2change], c = 'C0')
    ax_list_1[cent2change][0].scatter(c_coord_range, loss_acc[cent2change], c = region_colors[cent2change])

    ax_list_1[cent2change][0].set_xlabel(f"Centroid {cent2change}")
    
       
## Draw how assignment changes and how it affecst loss function
for i in range(len(c_coord_range)):
    for cent2change in range(kmeans.num_centroids): 
        
        # get associated coordinates, loss and regions per centroid.
        _c,_loss,_cent_assigned = c_coord_range[i],loss_acc[cent2change][i],assigned_regions[cent2change][i]
        
        ax_list_1[cent2change][1].cla()

        # draw data assignment
        colors_data = [f"C{i}" for i in _cent_assigned]     
        ax_list_1[cent2change][1].scatter(X, np.zeros_like(X), marker = 'x',  c = colors_data)

        # highlight loss
        ax_list_1[cent2change][0].plot( _c, _loss, 'o' ,c = 'k')#, markersize =10)
        
        # remove previous highlighted
        if i != 0:
            ax_list_1[cent2change][0].plot(  c_coord_range[i-1], loss_acc[cent2change][i-1], 'o' , c = region_colors[cent2change][i-1])

        ## Get centroids from the algorithm
        centroids = kmeans.centroids

        # centroid
        centroids[cent2change,:] = _c

        ## draw centroid
        for idx,cet in enumerate(centroids):
            ## plot centroids
            ax_list_1[cent2change][1].plot(cet,0.0, 'o', color = f'C{idx}')

        ## draw voronoi regions
        voronoi_regions, _ = assign_data_to_centroids(vor_grid_xx , centroids)
        voronoi_regions_limits = np.where(np.diff(voronoi_regions) != 0)[0] + 1

        for c in voronoi_regions_limits:
            ax_list_1[cent2change][1].axvline(vor_grid_xx[c], color='k', linestyle="--")
            
        # fix xlim
        ax_list_1[cent2change][1].set_xlim([grid_lim_l, grid_lim_u])
        

    fig_1.canvas.draw()
    frame = np.asarray(fig_1.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)

writer.close()     

display(Video(data=video_filename, embed=True))
os.remove(video_filename)
plt.close(fig_1)

In [ ]:
interactive_plot = False

if interactive_plot:
    %matplotlib tk
    plt.close("all")
else:
    %matplotlib inline
    
plotter = Plot1DKmeans(
                        video = True, 
                        interactive_plot = interactive_plot, 
                        sleep_time = 0.01, 
                        draw_voronoi = True,
                        loss_fun_optim = True,
                        num_loss_fun_centroids_to_show = 3 
                      )

kmeans = Kmeans(num_centroids = 3, plotter = plotter)
kmeans.run(X, num_iters = 10, seed = 10)

plotter.show_video()
print(kmeans.centroids)

**Change initialization**

In [ ]:
interactive_plot = False

if interactive_plot:
    %matplotlib tk
    plt.close("all")
else:
    %matplotlib inline
    
plotter = Plot1DKmeans(
                        video = True, 
                        interactive_plot = interactive_plot, 
                        sleep_time = 0.01, 
                        draw_voronoi = True,
                        loss_fun_optim = True,
                        num_loss_fun_centroids_to_show = 3 
                      )

kmeans = Kmeans(num_centroids = 3, plotter = plotter)
kmeans.run(X, num_iters = 10, seed = 1)

plotter.show_video()
print(kmeans.centroids)

#### 2 dimensional data

In [ ]:
np.random.seed(1)

N_points = 100

## probability p(c)
p = [0.33,0.33,0.34]
c = [0,1,2]

## probability p(x|c)
p_xc = {
    'c_0' : {
        'mu' : np.array([0,0]),
        'cov' : 0.05*np.eye(2),
        },
    'c_1' : {
        'mu' : np.array([0.5,0.5]),
        'cov' : 0.05*np.eye(2),
        },
    'c_2' : {
        'mu' : np.array([0,0.5]),
        'cov' : 0.05*np.eye(2),
        }   
}

## sample cluster assignments
cluster = np.random.choice(c, size=N_points, p=p)

## 
X = np.zeros((N_points,2), dtype = np.float32)
counter = 0
for _c in c:
    num_c = np.sum(cluster==_c)
    
    mu  = p_xc[f'c_{_c}']['mu']
    cov = p_xc[f'c_{_c}']['cov']
    
    _x = np.random.multivariate_normal(mean=mu, cov=cov, size = num_c)
    
    X[counter:counter + num_c] = _x
    X_clust[_c] = _x
    
    counter += num_c


fig, (ax1,ax2) = plt.subplots(1,2, figsize = (10,5))
ax1.plot(X[:,0],X[:,1], 'x', color = 'k')
ax1.set_xlabel(r"$x_1$")
ax1.set_ylabel(r"$x_2$")
ax1.set_title("Unlabelled Data")

for _c in c:
    _x = X_clust[_c]
    ax2.plot(_x[:,0],_x[:,1], 'x', color = f"C{_c}")
    ax2.set_xlabel(r"$x_1$")
    ax2.set_ylabel(r"$x_2$")
    ax2.set_title("True Cluster Assignment")

## shuffle data
for i in range(10):
    np.random.shuffle(X)

In [ ]:
interactive_plot = False

if interactive_plot:
    %matplotlib tk
    plt.close("all")
else:
    %matplotlib inline
    
plotter = Plot2DKmeans(
                        video = True, 
                        interactive_plot = interactive_plot, 
                        sleep_time = 0.01, 
                        draw_voronoi = True,
                        loss_fun_optim = True,
                        num_loss_fun_centroids_to_show = 4
                      )

kmeans = Kmeans(num_centroids = 4, plotter = plotter)
kmeans.run(X, num_iters = 20, seed = 1)

plotter.show_video()

### Gaussian Mixture Model via EM  approach

* To be done.

- Also performs coordinate descent but over probability distributions and parameters.
- Starts from a different loss function. Need to check if equivalent to the one above. Even if the algorithm is the same, it does not imply that the loss function form might be the same, only the minima and their locations.

 

### Model Selection

#### Choosing the number of clusters K: Elbow method 

A fundamental difficulty in K-means is model selection, that is, choosing the number of
clusters $K$.

As $K$ increases, the value of the loss function (within-cluster variance) always decreases.
In the extreme case where $K = N$, the loss becomes zero, even though the resulting clustering
is not meaningful.

Therefore, minimizing the loss function alone is not sufficient to determine the optimal
number of clusters.


##### For selecting the optimal number of clusters

The Elbow Method is a commonly used heuristic to select the number of clusters in K-means.

The idea is to analyze how the Within-Cluster Sum of Squares (WCSS) changes as the number of
clusters increases. WCSS measures how close the data points are to their corresponding
cluster centroids.

Formally, WCSS is defined as:

$$
\text{WCSS} =
\sum_{i=1}^{K}
\sum_{x_j \in S_i}
\text{distance}(x_j, c_i)^2
$$

where $\text{distance}(x_j, c_i)$ represents the distance between a data point $x_j$ and the
centroid $c_i$ of cluster $i$.

##### How the Elbow Method works

The Elbow Method follows these steps:

1. Select a range of values for $K$ (for example, from 1 to 10).
2. For each value of $K$, apply the K-means clustering algorithm and calculate the WCSS (Within-Cluster Sum of Squares).
3. Plot the curve of WCSS as a function of the number of clusters.
4. Observe how WCSS decreases as $K$ increases.

WCSS always decreases when adding more clusters, since clusters become smaller and more
compact. However, after a certain point, the improvement becomes very small.

This point appears as a bend or "elbow" in the curve.

- Before the elbow: WCSS decreases rapidly -> clusters improve significantly.
- After the elbow: WCSS decreases slowly -> additional clusters add little value and may
  lead to overfitting.

The value of $K$ corresponding to the elbow is considered a reasonable choice for the
number of clusters.

In [ ]:
# Range of K values
K_values = range(1, 10)
inertias = []

x1 = np.array([3, 1, 1, 2, 1, 6, 6, 6, 5, 6, 7, 8, 9, 8, 9, 9, 8, 4, 4, 5, 4])
x2 = np.array([5, 4, 5, 6, 5, 8, 6, 7, 6, 7, 1, 2, 1, 2, 3, 2, 3, 9, 10, 9, 10])

X = np.column_stack((x1, x2))

# Compute inertia (WCSS) for each K
for k in K_values:
    kmeans = KMeans_sklearn(n_clusters=k, init="k-means++", n_init=10, random_state=42)
    kmeans.fit(X)
    inertias.append(kmeans.inertia_)

# Chosen elbow point (based on visual inspection)
elbow_k = 4

# Plot
plt.figure()
plt.plot(K_values, inertias, marker='x')
plt.axvline(x=elbow_k, linestyle='--', label='Elbow Point', color = "red")
plt.xlabel("Number of clusters (k)")
plt.ylabel(" Inertia (WCSS)")
plt.title("The Elbow Method")
plt.legend()
plt.show()

The dataset used in this notebook is synthetic and is only intended for illustrative purposes, to visualize the behavior of the K-means algorithm and the Elbow Method.

We can apply the elbow method to the given dataset, and then we can identify an optimal value of K  for the K-means clustering algorithm. If we plot the WCSS for different values of K, we will get the plot above. The location of the bend in the plot is generally considered an indicator of the approximate number of clusters. 

Note that there is a steep decrease in the Within-Cluster Sum of Squares when K is increased from one to four, but much less when K is increased from four to nine. 


**Expected pattern:**

$\text{WSS}(K)$ decreases as $K$ increases (the groups are smaller).
At some point, the reduction in $\text{WSS}(K)$ becomes marginal. This point is the "elbow" and usually indicates a good value for $K$.

The "**elbow**" represents the point beyond which increasing $K$ does not
provide a significant improvement in the internal compactness
of the clusters.

##### Distortion and inertia

Two quantities commonly used in practice when applying the Elbow Method are distortion and
inertia.

**Distortion** measures the average squared distance between each data point and its assigned
cluster center:

$$
\text{Distortion} =
\frac{1}{n}
\sum_{i=1}^{n}
\min_{c \in \text{clusters}} \| x_i - c \|^2
$$

**Inertia** corresponds to the total squared distance of each data point to its closest
cluster center:

$$
\text{Inertia} =
\sum_{i=1}^{n} \| x_i - c_i^* \|^2
$$

In scikit-learn, inertia is directly provided by the attribute `inertia_` of the KMeans
object and is equivalent to WCSS.

##### Implementation of Elbow Method - Ilustrative Example

The specific dataset used here is only illustrative.

In [ ]:
x1 = np.array([3, 1, 1, 2, 1, 6, 6, 6, 5, 6,
               7, 8, 9, 8, 9, 9, 8, 4, 4, 5, 4])
x2 = np.array([5, 4, 5, 6, 5, 8, 6, 7, 6, 7,
               1, 2, 1, 2, 3, 2, 3, 9, 10, 9, 10])

X = np.column_stack((x1, x2))

plt.scatter(X[:, 0], X[:, 1])
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Dataset")
plt.show()

Although the visualization suggests the presence of approximately three clusters, 
visual inspection alone does not provide a principled way to select the optimal number 
of clusters. Therefore, we turn to a more systematic analysis.

In [ ]:
K_values = range(1, 8)
inertias = []

for k in K_values:
    kmeans = KMeans_sklearn(n_clusters=k, init="k-means++", n_init=10, random_state=42)
    kmeans.fit(X)
    inertias.append(kmeans.inertia_)

plt.plot(K_values, inertias, marker="o")
plt.xlabel("Number of clusters K")
plt.ylabel("Inertia (WCSS)")
plt.title("Elbow Method")
plt.show()

As shown in the clustering visualizations above, a clear change in slope can be observed around K = 4.
Up to this point, increasing the number of clusters leads to a substantial
reduction in inertia. Beyond K = 4, further decreases become marginal,
indicating diminishing returns.

Therefore, K = 4 appears to be a reasonable choice for the number of clusters
in this dataset according to the Elbow Method.

In [ ]:
for k in range(1,5):
    kmeans = KMeans_sklearn(n_clusters=k, init="k-means++", n_init=10, random_state=42)
    labels = kmeans.fit_predict(X)

    plt.scatter(X[:, 0], X[:, 1], c=labels)
    plt.scatter(
        kmeans.cluster_centers_[:, 0],
        kmeans.cluster_centers_[:, 1],
        marker="x",
        s=200,
        label="Centroids"
    )
    plt.title(f"K-means clustering with K = {k}")
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.legend()
    plt.show()

By comparing the clustering results for different values of K, we can clearly
observe how the structure of the data is progressively captured as the number
of clusters increases.

For K = 1, all data points are forced into a single cluster, leading to very
high within-cluster variance and a clear underfitting of the data.

With K = 2, the algorithm separates the data into two broad groups. Although
this reduces the within-cluster variance, the clusters are still too coarse
and fail to capture the finer structure present in the dataset.

For K = 3, the clustering improves significantly, with more compact clusters.
However, some groups remain heterogeneous, suggesting that relevant structure
is still being merged.

Finally, with K = 4, the algorithm produces compact and well-separated clusters,
with centroids located near the natural centers of each group. The clusters are
coherent, and no meaningful group appears to be artificially merged or split.

This qualitative analysis is consistent with the Elbow Method, where the
reduction in WCSS starts to flatten after K = 4. Therefore, K = 4 represents
a good trade-off between model simplicity and clustering quality.

##### General observations on the Elbow Method and K-means behavior

The previous visualizations illustrate how the K-means algorithm behaves as the
number of clusters K increases.

For small values of K, clusters tend to be overly coarse, leading to high
within-cluster variance and underfitting. As K increases, clusters become more
compact and better separated, which reduces the objective function optimized
by K-means.

However, beyond a certain point, increasing K yields diminishing returns:
the reduction in within-cluster variance becomes progressively smaller, while
model complexity continues to grow. This behavior motivates the use of the
Elbow Method as a heuristic criterion to select a reasonable number of clusters.

Rather than providing an exact optimal value, the Elbow Method highlights a
range of K values where the trade-off between model complexity and clustering
quality is balanced. This criterion is especially useful in unsupervised
settings, where ground truth labels are not available and standard validation
metrics cannot be directly applied.


##### Notes on sources and attribution 

The explanation of the Elbow Method follows the standard presentation commonly
found in introductory machine learning resources and is largely inspired by the explanation provided in GeeksforGeeks:
https://www.geeksforgeeks.org/machine-learning/elbow-method-for-optimal-value-of-k-in-kmeans/

All simulations use synthetic data generated solely for illustrative purposes.

## TODO
* Explanation of the coordinate descent algorithm; proof that k-means always converges
* K-means algorithm via EM and Gaussian mixtures
* Show that the cost function is not convex.
* Implement k-means++
* Showing the connection between within-class variance and between-cluster standard deviations.
* For the 2-dimensional data, in the loss function visualization, the different regions obtained when a cluster location varies and the cluster assignment varies could be highlighted in different colors as well, although it can be inspected from the bumps in the loss function.

## Contributors

**Model Selection - Elbow Method done by**


**Author:** Adriana Díaz Cano  
**Course:** Machine Learning  
**Degree:** Double Degree in Business Administration and Computer Engineering  
**University:** CUNEF Universidad  
**Year:** 2025-2026